# Fake News Detection

Classify a news article as **reliable (0)** or **fake (1)** from its headline and text.


**Steps:** clean text → remove stopwords → Porter stemming → TF-IDF → Logistic Regression.

In [2]:
import re
from functools import lru_cache

import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nltk.download('stopwords', quiet=True)
STOP_WORDS = set(stopwords.words('english'))

## 1. Load the data

In [3]:
df = pd.read_csv("./data/train.csv")
print("shape:", df.shape)
print(df.isnull().sum())
df.head()

shape: (20800, 5)
id           0
title      558
author    1957
text        39
label        0
dtype: int64


,id,title,author,text,label
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1


## 2. Leakage check: the `author` column

Does any column give the answer away without reading the article?

In [4]:
no_author = df["author"].isnull()
print("articles with no author:", no_author.sum())
print("fake among those       :", round(df[no_author]["label"].mean() * 100, 1), "%")
print("fake overall           :", round(df["label"].mean() * 100, 1), "%")

articles with no author: 1957
fake among those       : 98.7 %
fake overall           : 50.1 %


Almost every article with no author is fake, so the author name would let the model cheat. **The model is not given the author.**

## 3. Clean the rows

In [5]:
df["title"] = df["title"].fillna("")
df["text"] = df["text"].fillna("")
df["content"] = (df["title"] + " " + df["text"]).str.strip()

# remove duplicate articles, so the same article cannot be in both train and test
print("rows before:", len(df))
df = df.drop_duplicates(subset="content").drop_duplicates(subset="text").reset_index(drop=True)
print("rows after :", len(df))

rows before: 20800
rows after : 20387


## 4. Clean the text

Keep letters only → lowercase → drop stopwords → stem each word (`running` → `run`).

In [6]:
port_stem = PorterStemmer()

@lru_cache(maxsize=None)          # remember each word's stem, because words repeat a lot
def stem_word(word):
    return port_stem.stem(word)

def stemming(content):
    words = re.sub('[^a-zA-Z]', ' ', content).lower().split()
    return ' '.join(stem_word(word) for word in words if word not in STOP_WORDS)

df["clean"] = df["content"].apply(stemming)
print(df["clean"][1][:150])

flynn hillari clinton big woman campu breitbart ever get feel life circl roundabout rather head straight line toward intend destin hillari clinton rem


## 5. Train / test split

80% to train, 20% kept aside for the final test.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean"], df["label"], test_size=0.2, stratify=df["label"], random_state=42)

print("train:", len(X_train), " test:", len(X_test))

train: 16309  test: 4078


## 6. Train the model

TF-IDF turns each article into word weights (high if a word is common in this article but rare overall).
Logistic Regression learns one weight per word. 5-fold cross-validation picks the best `C` (regularisation strength).

In [ ]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(sublinear_tf=True, min_df=2, max_df=0.9)),
    ("clf", LogisticRegression(max_iter=2000)),
])

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = GridSearchCV(pipeline, {"clf__C": [0.1, 1, 10, 100]}, cv=folds, n_jobs=-1)
search.fit(X_train, y_train)

model = search.best_estimator_
print("best C:", search.best_params_["clf__C"])
print("cross-validation accuracy:", round(search.best_score_ * 100, 2), "%")

## 7. Test on the held-out 20%

In [ ]:
test_pred = model.predict(X_test)

print("train accuracy:", round(accuracy_score(y_train, model.predict(X_train)) * 100, 2), "%")
print("test accuracy :", round(accuracy_score(y_test, test_pred) * 100, 2), "%")
print()
print(classification_report(y_test, test_pred, target_names=["reliable", "fake"]))
print("confusion matrix (rows = true, columns = predicted):")
print(confusion_matrix(y_test, test_pred))

train accuracy: 100.0 %
test accuracy : 97.43 %

              precision    recall  f1-score   support

    reliable       0.98      0.97      0.97      2078
        fake       0.97      0.98      0.97      2000

    accuracy                           0.97      4078
   macro avg       0.97      0.97      0.97      4078
weighted avg       0.97      0.97      0.97      4078

confusion matrix (rows = true, columns = predicted):
[[2020   58]
 [  47 1953]]


## 8. Which words matter most?

In [ ]:
words = model["tfidf"].get_feature_names_out()
order = np.argsort(model["clf"].coef_[0])

print("reliable:", ", ".join(words[order[:15]]))
print("fake    :", ", ".join(words[order[-15:][::-1]]))

reliable: breitbart, mr, twitter, york, said, ms, follow, sunday, presid, saturday, inaugur, islam, new, march, time
fake    : octob, novemb, anti, non, co, hillari, self, sourc, comment, share, old, elect, com, us, al


`breitbart`, `york`, `mr`, `ms` point to *reliable*: most reliable articles here come from The New York Times and
Breitbart, and their headlines end with the publisher's name. The next section checks how much the score depends on that.

## 9. Two extra checks

Both reuse the same train/test rows as above.

In [ ]:
def test_accuracy(texts, clf):
    clf = clone(clf).fit(texts[X_train.index], y_train)      # fresh copy, so the main model is not changed
    return round(accuracy_score(y_test, clf.predict(texts[X_test.index])) * 100, 2)

# Check 1: how much would the author name have helped? (plain TF-IDF + Logistic Regression on headlines)
plain = Pipeline([("tfidf", TfidfVectorizer()), ("clf", LogisticRegression(max_iter=2000))])
title = df["title"].apply(stemming)
author_title = (df["author"].fillna("") + " " + df["title"]).apply(stemming)

print("headline only    :", test_accuracy(title, plain), "%")
print("headline + author:", test_accuracy(author_title, plain), "%")

headline only    : 92.45 %


headline + author: 97.67 %


In [ ]:
# Check 2: is the model only spotting the publisher's name? Remove the names and retrain the same model.
no_publisher = df["content"].str.replace(r"(?i)new york times|nytimes|breitbart", " ", regex=True).apply(stemming)

print("without publisher names:", test_accuracy(no_publisher, model), "%")

without publisher names: 96.49 %


Accuracy drops by about one point, so the model is mostly reading the article, not the publisher's name.

## 10. Try it on one article

In [ ]:
def predict_news(title, text):
    prob_fake = model.predict_proba([stemming(title + " " + text)])[0, 1]
    return ("FAKE" if prob_fake >= 0.5 else "reliable") + f" (probability of fake: {prob_fake:.2f})"

row = df.loc[X_test.index[0]]
print("headline  :", row["title"])
print("true label:", "fake" if row["label"] == 1 else "reliable")
print("model says:", predict_news(row["title"], row["text"]))

headline  : For Whites Sensing Decline, Donald Trump Unleashes Words of Resistance - The New York Times
true label: reliable
model says: reliable (probability of fake: 0.00)


**Limitations:** the articles are 2016 US political news, so the model learns that period's writing style and topics,
not whether a claim is true. It would score lower on articles from publishers it has never seen.